# 02 May 05 Linear Clip — LR & LR decay search

Фиксируем **LinearBidder bin clip** (как в старом «fixed baseline»), а через Optuna перебираем **скидку MDP для DQN (`dqn_gamma`)**, **стартовые lr** для DQN и RewardNet и **стратегии затухания learning rate**:

- **DQN `dqn_gamma`**: сетка **1.0, 0.999, 0.99** (дисконт по горизонту Q-learning).
- **DQN LR decay**: `ExponentialLR` (несколько множителей по LR), либо без scheduler, либо `CosineAnnealingWarmRestarts` с периодами 2000 / 8000 шагов обучения (один `scheduler.step` на один gradient step).
- **RewardNet**: без decay или `ExponentialLR` с $\gamma \in \{0.9995, 0.9999\}$.

Профиль: `may06_default_linear_clip_lr_scheduler_search` в `profiles.py` (`n_trials=36`). Число триалов можно переопределить константой ниже.

In [ ]:
import sys
import pickle
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = cwd
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess

In [ ]:
_linear_scr_fpa = Path(REPO_ROOT) / 'example_notebooks' / 'evaluate_baselines' / 'best_params' / 'fpa_baseline_n10_rndm_42' / 'linear_scr_FPA.pkl'
with _linear_scr_fpa.open('rb') as f:
    linear_tuned_params = pickle.load(f)
linear_tuned_params

{'coef': 0.023335213830958296,
 'lower_clip': 9,
 'upper_clip': 1,
 'factor': 3.4224852046754637}

In [ ]:
RUN_NAME = 'may06_clip_lr_scheduler_search'
DRLB_PROFILE = 'may05_default_clip_lr_scheduler_search'
VERBOSE = False
SHOW_PROGRESS = True
# None = взять n_trials из профиля (36)
N_TRIALS_OVERRIDE = None

In [ ]:
config = build_drlb_config(RUN_NAME, profile=DRLB_PROFILE, split_set='full_train_val_holdout')
if N_TRIALS_OVERRIDE is not None:
    config = replace(config, n_trials=int(N_TRIALS_OVERRIDE))
config = replace(config, refit_on='train_plus_val', max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])
base_drlb_params['bid_lower_clip'] = 3
base_drlb_params['bid_upper_clip'] = 8
base_drlb_params['traffic_path'] = str(REPO_ROOT / 'data' / 'traffic_share.csv')

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    show_progress=SHOW_PROGRESS,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
tuning = summary['tuning']
{
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'n_trials': tuning['n_trials'],
    'study_best_val_clicks': tuning['study_best_value'],
    'best_trial': tuning['best_trial_number'],
    'best_params': tuning['best_params'],
    'best_val_metrics': tuning['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}

autobidder_check campaigns: 100%|██████████| 257/257 [00:17<00:00, 14.49campaign/s, campaign_id=7.46e+7]
[I 2026-05-06 01:13:18,308] A new study created in memory with name: no-name-ab05ce48-e530-4d5f-a765-04e8ad1d3ff1
Best trial: 0. Best value: 1654.88:   3%|▎         | 1/36 [03:20<1:57:06, 200.77s/it]

[I 2026-05-06 01:16:39,075] Trial 0 finished with value: 1654.8796704626443 and parameters: {'dqn_gamma': 1.0, 'dqn_lr': 0.003, 'reward_net_lr': 0.01, 'dqn_lr_decay': 'cosine_T8000', 'reward_net_lr_decay': 'none'}. Best is trial 0 with value: 1654.8796704626443.


Best trial: 0. Best value: 1654.88:   6%|▌         | 2/36 [06:35<1:51:43, 197.16s/it]

[I 2026-05-06 01:19:53,699] Trial 1 finished with value: 1546.8655996726086 and parameters: {'dqn_gamma': 1.0, 'dqn_lr': 0.0001, 'reward_net_lr': 0.003, 'dqn_lr_decay': 'cosine_T8000', 'reward_net_lr_decay': 'exp_0.9995'}. Best is trial 0 with value: 1654.8796704626443.


Best trial: 0. Best value: 1654.88:   8%|▊         | 3/36 [09:53<1:48:44, 197.72s/it]

...

Best trial: 6. Best value: 2070.55:  39%|███▉      | 14/36 [7:57:45<37:07:27, 6074.88s/it]

[I 2026-05-06 09:11:03,476] Trial 13 finished with value: 1885.7283864153887 and parameters: {'dqn_gamma': 0.999, 'dqn_lr': 0.003, 'reward_net_lr': 0.001, 'dqn_lr_decay': 'exp_0.9995', 'reward_net_lr_decay': 'exp_0.9999'}. Best is trial 6 with value: 2070.5456830456856.


Best trial: 6. Best value: 2070.55:  39%|███▉      | 14/36 [7:59:15<12:33:06, 2053.95s/it]


[W 2026-05-06 09:12:33,596] Trial 14 failed with parameters: {'dqn_gamma': 0.99, 'dqn_lr': 0.0003, 'reward_net_lr': 0.001, 'dqn_lr_decay': 'cosine_T2000', 'reward_net_lr_decay': 'exp_0.9999'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/amsafin/code/local_ml/rl/bat_venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/adapters/drlb_adapter.py", line 144, in optuna_objective
    run = run_drlb_candidate(
          ^^^^^^^^^^^^^^^^^^^
  File "/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/adapters/drlb_adapter.py", line 354, in run_drlb_candidate
    bidder.fit(
  File "/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/simulator/model/drlb_bidder.py", line 524, in fit
    self.agent.learn_d

KeyboardInterrupt: 

In [ ]:
trials_df = pd.DataFrame(tuning['all_trials_summary'])
if 'clicks_sum' in trials_df.columns:
    trials_df = trials_df.sort_values('clicks_sum', ascending=False)
trials_df

,trial,dqn_gamma,dqn_lr,reward_net_lr,dqn_lr_decay,reward_net_lr_decay,rmse,clicks_sum,cpc_relative,quickspend,last_dqn_loss,last_reward_net_loss,dqn_loss_mean,reward_net_loss_mean
26,26,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
33,33,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
32,32,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
31,31,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
24,24,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
25,25,1.000,0.0003,0.0100,cosine_T2000,exp_0.9999,1.249205,2466.061439,1051.546352,0.023346,46.399685,139.428360,23.377484,240.922805
8,8,1.000,0.0003,0.0010,cosine_T2000,exp_0.9999,1.436881,2391.657496,818.878595,0.054475,25.303316,257.954254,21.704505,250.602151
34,34,1.000,0.0003,0.0030,cosine_T8000,exp_0.9999,1.255702,2340.115990,1207.533119,0.027237,24.510843,212.278915,26.648206,197.959975
18,18,1.000,0.0010,0.0100,cosine_T2000,exp_0.9999,1.401908,2326.004889,974.469456,0.058366,28.818624,117.121498,16.557584,255.321782
22,22,1.000,0.0010,0.0100,cosine_T2000,exp_0.9999,1.401908,2326.004889,974.469456,0.058366,28.818624,117.121498,16.557584,255.321782


In [ ]:
pd.DataFrame([
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
])

,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
